In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

In [2]:

DATA_PATH = r"C:\Users\Vineet\OneDrive\Desktop\credit_card_resume\data\creditcard.csv"
data = pd.read_csv(DATA_PATH)

X = data.drop(columns="Class")
y = data["Class"]

# Same split and folds as the baseline comparison
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("Train frauds:", y_train.sum(), "| Test frauds:", y_test.sum())

X_train: (227845, 30) | X_test: (56962, 30)
Train frauds: 394 | Test frauds: 98


In [3]:
rf_grid = {
    "n_estimators": [100, 200],
    "max_depth": [10, 20, None],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", 0.3],
    "class_weight": [None, "balanced_subsample"],
    "max_samples": [0.3, 0.5],
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    rf_grid,
    n_iter=4,                  # 4 configs x 5 folds = 20 fits
    scoring="average_precision",
    cv=cv,
    random_state=42,
    n_jobs=1,
    verbose=2,
)
rf_search.fit(X_train, y_train)

best_rf = rf_search.best_estimator_
i = rf_search.best_index_

print("Best params:", rf_search.best_params_)
print(f"Best CV PR-AUC: {rf_search.cv_results_['mean_test_score'][i]:.4f} "
      f"± {rf_search.cv_results_['std_test_score'][i]:.4f}")

Fitting 5 folds for each of 4 candidates, totalling 20 fits
[CV] END class_weight=balanced_subsample, max_depth=20, max_features=0.3, max_samples=0.5, min_samples_leaf=2, n_estimators=200; total time=  28.4s
[CV] END class_weight=balanced_subsample, max_depth=20, max_features=0.3, max_samples=0.5, min_samples_leaf=2, n_estimators=200; total time=  25.0s
[CV] END class_weight=balanced_subsample, max_depth=20, max_features=0.3, max_samples=0.5, min_samples_leaf=2, n_estimators=200; total time=  22.7s
[CV] END class_weight=balanced_subsample, max_depth=20, max_features=0.3, max_samples=0.5, min_samples_leaf=2, n_estimators=200; total time=  22.9s
[CV] END class_weight=balanced_subsample, max_depth=20, max_features=0.3, max_samples=0.5, min_samples_leaf=2, n_estimators=200; total time=  22.5s
[CV] END class_weight=None, max_depth=10, max_features=0.3, max_samples=0.5, min_samples_leaf=1, n_estimators=200; total time=70.6min
[CV] END class_weight=None, max_depth=10, max_features=0.3, max_sa

In [4]:
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    precision_score, recall_score, f1_score, confusion_matrix
)

probs = best_rf.predict_proba(X_test)[:, 1]
preds = (probs >= 0.5).astype(int)

print(f"PR-AUC    : {average_precision_score(y_test, probs):.4f}")
print(f"ROC-AUC   : {roc_auc_score(y_test, probs):.4f}")
print(f"Precision : {precision_score(y_test, preds):.4f}")
print(f"Recall    : {recall_score(y_test, preds):.4f}")
print(f"F1        : {f1_score(y_test, preds):.4f}")

tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
print(f"\nTN={tn}  FP={fp}  FN={fn}  TP={tp}")

PR-AUC    : 0.8528
ROC-AUC   : 0.9654
Precision : 0.8916
Recall    : 0.7551
F1        : 0.8177

TN=56855  FP=9  FN=24  TP=74
